# Code for coverage analysis: #

The pipeline for analysis is:
1. Load satellite states (Given in MCI)
2. Convert satellite states from MCI to PA 
3. Convert the user position in PA
4. Compute the line-of-sight vector
5. Convert the PA line-of-sight into the ENU frame (East, North, Up)
6. Compute the corresponding elevation
7. Apply elevation mask to remove satellites
8. Compute DOP and other values of interest

### Step 1: Load Satellite States: ###

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

######## Sub-step 1 - Loading the CSV files:
# Folder containing the CSV files
data_dir = Path("")

# Find all satellite CSV files
csv_files = sorted(data_dir.glob("a19056-i60_Satellite_*.csv"))

print(f"Found {len(csv_files)} satellite files:")
for file in csv_files:
    print(file.name)


######## Sub-step 2 - Saving the CSV files to a dictionary:
sat_data = {}
for file in csv_files:
    # Extract satellite number from filename
    # Example: S-ELFO_Satellite_05.csv -> "05"
    sat_id = file.stem.split("_")[-1]

    df = pd.read_csv(file)

    sat_data[sat_id] = df

######## Sub-step 3 - Extracting the corresponding data: 
sat_states_mci = {}
for sat_id, df in sat_data.items():
    # Time arrays
    t_et = df["EphermerisTime"].to_numpy()
    t_elapsed = df["ElapsedTime"].to_numpy()

    # Position in MCI
    r_mci = df[["x_MCI", "y_MCI", "z_MCI"]].to_numpy()

    # Velocity in MCI
    v_mci = df[["vx_MCI", "vy_MCI", "vz_MCI"]].to_numpy()

    # Full 6-state in MCI
    x_mci = df[["x_MCI", "y_MCI", "z_MCI", 
                "vx_MCI", "vy_MCI", "vz_MCI"]].to_numpy()

    sat_states_mci[sat_id] = {
        "t_et": t_et,
        "t_elapsed": t_elapsed,
        "r_mci": r_mci,
        "v_mci": v_mci,
        "x_mci": x_mci
    }

### Step 2: Convert MCI to PA: ###

In [ ]:
def mci_to_pa(t_tdb, rv_mci):
    """
    Convert state vector from MCI to PA frame at given time.
    For simplicity, we assume MCI and PA frames are aligned at t=0 and do not precess.
    In reality, you would need to account for the rotation of the Moon and the orientation of the PA frame.

    Parameters:
        t_tdb: Time in seconds since epoch J2000 (TDB)
        rv_mci: State vector in MCI frame [x, y, z, vx, vy, vz]
    """
    rv_pa = np.zeros_like(rv_mci)  # placeholder for the actual MCI state vector
    phi, theta, psi, phi_dot, theta_dot, psi_dot = get_lunar_angles(t_tdb)
    
    # Obtain the position and velocity from the given rv_mci variable
    rv_mci = np.asarray(rv_mci, dtype=float)
    r_mci = rv_mci[:3]          # Position
    v_mci = rv_mci[3:]          # Velocity

    # Find the TRANSFORMATION MATRIX for the position
    # R_total = R3_psi * R1_theta * R3_phi
    R3_psi = R3_matrix(psi)
    R1_theta = R1_matrix(theta)
    R3_phi = R3_matrix(phi)
    R_total = R3_psi @ R1_theta @ R3_phi

    # Obtain the new position using the computed matrix in PA frame
    r_new_PA = R_total @ r_mci

    # Find the R_dot matrix for the velocity
    R_total_dot = (R3_matrix_dot(psi, psi_dot) @ R1_theta @ R3_phi +
      R3_psi @ R1_matrix_dot(theta, theta_dot) @ R3_phi + R3_psi @ R1_theta @
      R3_matrix_dot(phi, phi_dot))

    # Compute the new velocity in PA
    v_new_PA = R_total @ v_mci + R_total_dot @ r_mci

    # Combine the value for the final array
    rv_pa = np.hstack((r_new_PA, v_new_PA))

    # Return the values
    return rv_pa

# For the first rotation matrix:
def R1_matrix(a):
    # Define the numpy cosine and sine:
    c = np.cos(a)
    s = np.sin(a)

    # Define the R1 rotation:
    R1 = np.array(([1, 0, 0],
                   [0, c, s],
                   [0, -s, c]))
    return R1

# For the third rotation matrix:
def R3_matrix(a):
    # Define the numpy cosine and sine:
    c = np.cos(a)
    s = np.sin(a)

    # Define the R3 rotation:
    R3 = np.array(([c, s, 0],
                   [-s, c, 0],
                   [0, 0, 1]))
    return R3

# For the first rotation matrix derivative:
def R1_matrix_dot(a, a_dot):
    # Define the numpy cosine and sine:
    c = np.cos(a)
    s = np.sin(a)

    # Define the R1 rotation derivative:
    R1_dot = np.array(([0, 0, 0],
                       [0, -a_dot*s, a_dot*c],
                       [0, -a_dot*c, -a_dot*s]))
    return R1_dot

# For the third rotation matrix derivative:
def R3_matrix_dot(a, a_dot):
    # Define the numpy cosine and sine:
    c = np.cos(a)
    s = np.sin(a)

    # Define the R3 rotation derivative:
    R3_dot = np.array(([-a_dot*s, a_dot*c, 0],
                       [-a_dot*c, -a_dot*s, 0],
                       [0, 0, 0]))
    return R3_dot

def get_lunar_angles(t_tdb):
    """
    Approximate lunar orientation angles based on Archinal et al. (2011), Table 2.
    https://aa.usno.navy.mil/downloads/reports/Archinaletal2011a.pdf#page=5.87

    Parameters
    ----------
    t_tdb : float
        Seconds since J2000 epoch (TDB), where J2000 = JD 2451545.0.

    Returns
    -------
    phi, theta, psi : float
        3-1-3 Euler angles [rad], using the convention
            R_mci_to_pa = Rz(psi) @ Rx(theta) @ Rz(phi)

    phi_dot, theta_dot, psi_dot : float
        Time derivatives of the Euler angles [rad/s]
    """

    # Time variables used in the IAU/Archinal formulas
    d = t_tdb / 86400.0              # days from J2000
    T = d / 36525.0                  # Julian centuries from J2000

    # E_i arguments in degrees: E_i = E0_i + Edot_i * d
    E0 = np.array([
        125.045, 250.089, 260.008, 176.625, 357.529,
        311.589, 134.963, 276.617,  34.226,  15.134,
        119.743, 239.961,  25.053
    ], dtype=float)

    Edot = np.array([
        -0.0529921, -0.1059842, 13.0120009, 13.3407154,  0.9856003,
        26.4057084, 13.0649930,  0.3287146,  1.7484877, -0.1589763,
         0.0036096,  0.1643573, 12.9590088
    ], dtype=float)   # deg/day

    E_deg = E0 + Edot * d
    E_rad = np.deg2rad(E_deg)
    Edot_rad_per_day = np.deg2rad(Edot)

    # Coefficients from the table
    # alpha0 = 269.9949 + 0.0031 T + sum_i a_i sin(E_i)
    a_alpha = np.array([
        -3.8787, -0.1204,  0.0700, -0.0172,  0.0,
         0.0072,  0.0,     0.0,     0.0,    -0.0052,
         0.0,     0.0,     0.0043
    ], dtype=float)

    # delta0 = 66.5392 + 0.0130 T + sum_i a_i cos(E_i)
    a_delta = np.array([
         1.5419,  0.0239, -0.0278,  0.0068,  0.0,
        -0.0029,  0.0009,  0.0,     0.0,     0.0008,
         0.0,     0.0,    -0.0009
    ], dtype=float)

    # W = 38.3213 + 13.17635815 d - 1.4e-12 d^2 + sum_i a_i sin(E_i)
    a_W = np.array([
         3.5610,  0.1208, -0.0642,  0.0158,  0.0252,
        -0.0066, -0.0047, -0.0046,  0.0028,  0.0052,
         0.0040,  0.0019, -0.0044
    ], dtype=float)

    # Evaluate alpha0, delta0, W in degrees
    alpha0_deg = 269.9949 + 0.0031 * T + np.sum(a_alpha * np.sin(E_rad))
    delta0_deg =  66.5392 + 0.0130 * T + np.sum(a_delta * np.cos(E_rad))
    W_deg      =  38.3213 + 13.17635815 * d - 1.4e-12 * d**2 + np.sum(a_W * np.sin(E_rad))

    # Time derivatives with respect to d [deg/day]
    dalpha_dd = (
        0.0031 / 36525.0
        + np.sum(a_alpha * np.cos(E_rad) * Edot_rad_per_day)
    )

    ddelta_dd = (
        0.0130 / 36525.0
        - np.sum(a_delta * np.sin(E_rad) * Edot_rad_per_day)
    )

    dW_dd = (
        13.17635815
        - 2.8e-12 * d
        + np.sum(a_W * np.cos(E_rad) * Edot_rad_per_day)
    )

    # Map (alpha0, delta0, W) to 3-1-3 Euler angles
    # Standard choice for inertial -> body-fixed
    phi_deg   = 90.0 + alpha0_deg
    theta_deg = 90.0 - delta0_deg
    psi_deg   = W_deg

    # Corresponding rates [deg/day]
    dphi_dd   = dalpha_dd
    dtheta_dd = -ddelta_dd
    dpsi_dd   = dW_dd

    # Convert to radians and radians/second
    phi   = np.deg2rad(phi_deg)
    theta = np.deg2rad(theta_deg)
    psi   = np.deg2rad(psi_deg)

    phi_dot   = np.deg2rad(dphi_dd) / 86400.0
    theta_dot = np.deg2rad(dtheta_dd) / 86400.0
    psi_dot   = np.deg2rad(dpsi_dd) / 86400.0

    return phi, theta, psi, phi_dot, theta_dot, psi_dot


# Convert all satellite states from MCI to PA
sat_states_pa = {}
for sat_id, data in sat_states_mci.items():
    t_et = data["t_et"]
    x_mci = data["x_mci"]

    # Allocate array for PA states
    x_pa = np.zeros_like(x_mci)

    # Convert each time step from MCI to PA
    for k in range(len(t_et)):
        if k % 1000 == 0:
            print(f"Satellite {sat_id}: time step {k}/{len(t_et)}")
        x_pa[k] = mci_to_pa(t_et[k], x_mci[k])

    sat_states_pa[sat_id] = {
        "t_et": t_et,
        "t_elapsed": data["t_elapsed"],
        "x_pa": x_pa,
        "r_pa": x_pa[:, :3],
        "v_pa": x_pa[:, 3:]
    }

### Step 3: Compute User Position in PA: ###

In [ ]:
# Function to compute position
def get_user_pos_pa(lat_deg, lon_deg, R_moon=1737.4):
    # Take the lat and lon
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    # Compute user position in PA
    r_user_pa = R_moon * np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])

    # Return user position
    return r_user_pa

# Actual position computation
lat_user = -90.0   # south pole
lon_user = 0.0

# Get the user position
r_user_pa = get_user_pos_pa(lat_user, lon_user)

### Step 4: Compute the Line-of-Sight Vector: ###

In [ ]:
# Function to compute line of sight vector
def compute_los_pa(r_sat_pa, r_user_pa):
    # Vector from user to satellite
    rho_pa = r_sat_pa - r_user_pa

    # Range
    rho_norm = np.linalg.norm(rho_pa, axis=1)

    # Unit line-of-sight vector
    rho_hat_pa = rho_pa / rho_norm[:, None]

    return rho_pa, rho_hat_pa, rho_norm

# Find the line of sight for every satellite
sat_los_pa = {}
for sat_id, data in sat_states_pa.items():
    r_sat_pa = data["r_pa"]

    rho_pa, rho_hat_pa, rho_norm = compute_los_pa(r_sat_pa, r_user_pa)

    sat_los_pa[sat_id] = {
        "rho_pa": rho_pa,
        "rho_hat_pa": rho_hat_pa,
        "rho_norm": rho_norm
    }

### Step 5: PA Line-of-Sight Into the ENU Frame ###

In [ ]:
# Function for conversion
def get_rot_pa_to_enu(lat_deg, lon_deg):
    # Get the lat and lon
    lat = np.deg2rad(lat_deg)
    lon = np.deg2rad(lon_deg)

    e_hat = np.array([
        -np.sin(lon),
         np.cos(lon),
         0.0
    ])

    n_hat = np.array([
        -np.sin(lat) * np.cos(lon),
        -np.sin(lat) * np.sin(lon),
         np.cos(lat)
    ])

    u_hat = np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])

    # Assemble the hat vector
    R_pa_to_enu = np.vstack((e_hat, n_hat, u_hat))

    # Return the hat vector
    return R_pa_to_enu


# Apply line-of-sight for every location. Start with user position
R_pa_to_enu = get_rot_pa_to_enu(lat_user, lon_user)

# For the satellite
sat_los_enu = {}
for sat_id, data in sat_los_pa.items():
    rho_hat_pa = data["rho_hat_pa"]   
    
    # Since rho_hat_pa is stored row-by-row, use R.T for batch multiplication
    rho_hat_enu = rho_hat_pa @ R_pa_to_enu.T

    sat_los_enu[sat_id] = {
        "rho_hat_enu": rho_hat_enu
    }

### Step 6: Compute Elevation: ##

In [ ]:
# Function to compute the elevation and visibility
def compute_elevation_and_visibility(rho_hat_enu, elev_mask_deg=5.0):
    # Up component of the ENU unit vector
    u_component = rho_hat_enu[:, 2]

    # Clip for numerical safety, since arcsin requires values in [-1, 1]
    u_component = np.clip(u_component, -1.0, 1.0)

    # Elevation angle
    elevation_rad = np.arcsin(u_component)
    elevation_deg = np.rad2deg(elevation_rad)

    # Visibility condition
    is_visible = elevation_deg > elev_mask_deg

    return elevation_rad, elevation_deg, is_visible

# Compute the elevation and visibility for the system: 
elev_mask_deg = 5.0
sat_visibility = {}
for sat_id, data in sat_los_enu.items():
    rho_hat_enu = data["rho_hat_enu"]

    # Get the elevation and corresponding visibility status
    elevation_rad, elevation_deg, is_visible = compute_elevation_and_visibility(
        rho_hat_enu,
        elev_mask_deg=elev_mask_deg
    )

    # Add the data for the satellite
    sat_visibility[sat_id] = {
        "elevation_rad": elevation_rad,
        "elevation_deg": elevation_deg,
        "is_visible": is_visible
    }

## Step 7: Apply Visibility Mask and Plot: ##

In [ ]:
sat_ids = sorted(sat_visibility.keys())

# Find the satellite with the longest visibility array
sat_longest = max(
    sat_ids,
    key=lambda sat_id: len(sat_visibility[sat_id]["is_visible"])
)

N_max = len(sat_visibility[sat_longest]["is_visible"])

# Use the time history from the longest satellite
t_elapsed = sat_states_pa[sat_longest]["t_elapsed"]
t_hours = (t_elapsed - t_elapsed[0]) / 3600.0

# Create visibility matrix filled with NaNs
# Shape: (N_max, number of satellites)
visibility_matrix = np.full((N_max, len(sat_ids)), np.nan)

for j, sat_id in enumerate(sat_ids):
    is_visible = sat_visibility[sat_id]["is_visible"]

    N_sat = len(is_visible)

    # Convert Boolean visibility to float:
    # True -> 1.0, False -> 0.0
    visibility_matrix[:N_sat, j] = is_visible.astype(float)

# Number of visible satellites at each time step
visible_count = np.nansum(visibility_matrix, axis=1)

# Number of satellites that actually have data at each time step
valid_satellite_count = np.sum(~np.isnan(visibility_matrix), axis=1)

Code to plot the visibility:

In [ ]:
import matplotlib.pyplot as plt

######## Elevation vs Time: 4 plots with 3 satellites each

import numpy as np
import matplotlib.pyplot as plt

sat_ids = sorted(sat_visibility.keys())

# Split satellites into groups of 3
sat_groups = [sat_ids[i:i+3] for i in range(0, len(sat_ids), 3)]

fig, axes = plt.subplots(
    nrows=len(sat_groups),
    ncols=1,
    figsize=(12, 14),
    sharex=True,
    sharey=True
)

# If there is only one group, make axes iterable
if len(sat_groups) == 1:
    axes = [axes]

for ax, group in zip(axes, sat_groups):
    for sat_id in group:
        # Time for this satellite
        t_elapsed_sat = sat_states_pa[sat_id]["t_elapsed"]
        t_hours_sat = (t_elapsed_sat - t_elapsed_sat[0]) / 3600.0

        # Elevation for this satellite
        elevation_deg = sat_visibility[sat_id]["elevation_deg"]

        # Optional downsampling for cleaner plotting
        plot_stride = max(1, len(t_hours_sat) // 3000)

        ax.plot(
            t_hours_sat[::plot_stride],
            elevation_deg[::plot_stride],
            label=f"Sat {sat_id}",
            linewidth=0.8
        )

    # Plot elevation mask on each subplot
    ax.axhline(
        elev_mask_deg,
        linestyle="--",
        color="k",
        linewidth=1.0,
        label=f"{elev_mask_deg} deg mask"
    )

    ax.set_ylabel("Elevation [deg]")
    ax.set_ylim([-90, 90])
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=4, fontsize=8)

axes[-1].set_xlabel("Elapsed Time [hours]")

fig.suptitle(
    f"Elevation vs Time at lat={lat_user} deg, lon={lon_user} deg",
    fontsize=14
)

plt.tight_layout()
plt.show()



######## Number of Visible vs Time
plt.figure(figsize=(12, 4))
plt.plot(t_hours, visible_count)
plt.xlabel("Elapsed Time [hours]")
plt.ylabel("Number of Visible Satellites")
plt.title(f"Coverage Count at lat={lat_user} deg, lon={lon_user} deg")
plt.grid(True)
plt.tight_layout()
plt.show()

## Step 8: Compute DOP: ##

In [ ]:
# FUNCTION TO COMPUTE DOP FROM VISIBLE SATELLITES
def compute_dop_from_visible_los(rho_hat_enu_visible):
    N_visible = rho_hat_enu_visible.shape[0]
    # Need at least 4 satellites for 3D position + clock bias
    if N_visible < 4:
        return {
            "GDOP": np.nan,
            "PDOP": np.nan,
            "HDOP": np.nan,
            "VDOP": np.nan,
            "TDOP": np.nan
        }
    # Build geometry matrix
    G = np.column_stack([
        rho_hat_enu_visible,
        np.ones(N_visible)
    ])
    # Compute covariance-like geometry matrix
    # Use pinv instead of inv to avoid crashes when geometry is poor
    Q = np.linalg.pinv(G.T @ G)
    # Since columns are [E, N, U, clock]
    q_EE = Q[0, 0]
    q_NN = Q[1, 1]
    q_UU = Q[2, 2]
    q_tt = Q[3, 3]
    GDOP = np.sqrt(q_EE + q_NN + q_UU + q_tt)
    PDOP = np.sqrt(q_EE + q_NN + q_UU)
    HDOP = np.sqrt(q_EE + q_NN)
    VDOP = np.sqrt(q_UU)
    TDOP = np.sqrt(q_tt)
    return {
        "GDOP": GDOP,
        "PDOP": PDOP,
        "HDOP": HDOP,
        "VDOP": VDOP,
        "TDOP": TDOP
    }

# COMPUTE THE DOP VALUES
sat_ids = sorted(sat_visibility.keys())
# Find satellite with longest time history
sat_longest = max(
    sat_ids,
    key=lambda sat_id: len(sat_visibility[sat_id]["is_visible"])
)
N_max = len(sat_visibility[sat_longest]["is_visible"])
print("Using longest satellite:", sat_longest)
print("N_max:", N_max)
# Use time from longest satellite
t_elapsed = sat_states_pa[sat_longest]["t_elapsed"]
t_hours = (t_elapsed - t_elapsed[0]) / 3600.0
# Allocate arrays
GDOP = np.full(N_max, np.nan)
PDOP = np.full(N_max, np.nan)
HDOP = np.full(N_max, np.nan)
VDOP = np.full(N_max, np.nan)
TDOP = np.full(N_max, np.nan)
num_visible_for_dop = np.zeros(N_max, dtype=int)
for k in range(N_max):
    visible_los_list = []
    for sat_id in sat_ids:
        # Skip if this satellite does not have this time index
        if k >= len(sat_visibility[sat_id]["is_visible"]):
            continue
        # Check visibility
        if sat_visibility[sat_id]["is_visible"][k]:
            # Get ENU LOS vector at this time
            rho_hat_enu_k = sat_los_enu[sat_id]["rho_hat_enu"][k]
            visible_los_list.append(rho_hat_enu_k)
    num_visible_for_dop[k] = len(visible_los_list)
    # Need at least 4 visible satellites
    if len(visible_los_list) >= 4:
        rho_hat_enu_visible = np.array(visible_los_list)
        dop = compute_dop_from_visible_los(rho_hat_enu_visible)
        GDOP[k] = dop["GDOP"]
        PDOP[k] = dop["PDOP"]
        HDOP[k] = dop["HDOP"]
        VDOP[k] = dop["VDOP"]
        TDOP[k] = dop["TDOP"]
# PLOT ALL DOP ON ONE PLOT
plt.figure(figsize=(12, 6))
plt.plot(t_hours, GDOP, label="GDOP")
plt.plot(t_hours, PDOP, label="PDOP")
plt.plot(t_hours, HDOP, label="HDOP")
plt.plot(t_hours, VDOP, label="VDOP")
plt.plot(t_hours, TDOP, label="TDOP")
plt.xlabel("Elapsed Time [hours]")
plt.ylabel("DOP")
plt.title(f"DOP vs Time at lat={lat_user} deg, lon={lon_user} deg")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



# PLOT THE SUBPLOTS
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True)
dop_data = {
    "GDOP": GDOP,
    "PDOP": PDOP,
    "HDOP": HDOP,
    "VDOP": VDOP
}
axes_flat = axes.flatten()
for ax, (dop_name, dop_values) in zip(axes_flat, dop_data.items()):
    ax.plot(
        t_hours,
        dop_values,
        marker="o",
        markersize=2,
        linewidth=1.2,
        label=f"lat={lat_user} deg, lon={lon_user} deg"
    )
    ax.set_title(f"{dop_name} Over Time")
    ax.set_ylabel(dop_name)
    ax.grid(True, alpha=0.6)
    ax.legend(fontsize=8)
    # Optional: match the screenshot's vertical range
    ax.set_ylim([0, 20])
# x-axis labels only on bottom plots
axes[1, 0].set_xlabel("Time (hours)")
axes[1, 1].set_xlabel("Time (hours)")
plt.tight_layout()
plt.show()

In [ ]:
######## Whole-Body 3D Coverage Plot + South Pole Views ########
# This block computes coverage over the entire lunar surface.
# Coverage here means: number of satellites with elevation angle above elev_mask_deg.
# The top row shows the full body in 3D.
# The bottom row shows a clean 2D south-pole projection for the same three times.
# Added: less-cluttered numbered circle markers for four NASA south-pole sites + Apollo 11.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D  # Needed for 3D projection

# -----------------------------
# User-adjustable settings
# -----------------------------
R_body = 1737.4  # km, lunar radius

# Smaller spacing = smoother sphere but slower computation/plotting.
# Good quick test: 5 deg. Good final plot: 2 or 3 deg.
dlat_global = 0.5
dlon_global = 0.5

# Use the same elevation mask defined earlier in the notebook.
elev_mask_global_deg = elev_mask_deg

# These are the three time snapshots to show.
time_snapshot_names = ["Start", "Middle", "End"]

# -----------------------------
# Surface site markers
# -----------------------------
# Longitudes are treated as degrees East in the PA/body-fixed frame.
# The south-pole sites are numbered to reduce clutter.
site_locations = [
    {
        "name": "Nobile Rim 2",
        "short_label": "1",
        "lat_deg": -84.2,
        "lon_deg": 60.03,
        "color": "red",
        "marker": "o",
        "size": 45,
        "show_on_south_pole": True,
    },
    {
        "name": "Malapert Massif",
        "short_label": "2",
        "lat_deg": -86.0,
        "lon_deg": 2.06,
        "color": "deepskyblue",
        "marker": "o",
        "size": 45,
        "show_on_south_pole": True,
    },
    {
        "name": "Henson SSE Rima",
        "short_label": "3",
        "lat_deg": -89.1,
        "lon_deg": -109.3,
        "color": "limegreen",
        "marker": "o",
        "size": 45,
        "show_on_south_pole": True,
    },
    {
        "name": "Faustini Rim A",
        "short_label": "4",
        "lat_deg": -87.3,
        "lon_deg": 77.0,
        "color": "magenta",
        "marker": "o",
        "size": 45,
        "show_on_south_pole": True,
    },
    {
        # Apollo 11 / Tranquility Base
        "name": "Apollo 11",
        "short_label": "A11",
        "lat_deg": 0.6741,
        "lon_deg": 23.4729,
        "color": "gold",
        "marker": "*",
        "size": 120,
        "show_on_south_pole": False,
    },
]


def latlon_to_xyz(lat_deg, lon_deg, radius=1737.4):
    """
    Convert spherical latitude/longitude [deg] to Cartesian PA coordinates [km].
    """
    lat_rad = np.deg2rad(lat_deg)
    lon_rad = np.deg2rad(lon_deg)

    x = radius * np.cos(lat_rad) * np.cos(lon_rad)
    y = radius * np.cos(lat_rad) * np.sin(lon_rad)
    z = radius * np.sin(lat_rad)

    return x, y, z


def build_global_body_grid(dlat, dlon, R_body=1737.4):
    """
    Build a latitude/longitude grid over the full spherical body.

    Returns
    -------
    Lat_grid, Lon_grid : 2D arrays, degrees
    r_grid : 3D array, shape (N_lat, N_lon, 3)
        Surface position vectors in PA coordinates.
    up_hat_grid : 3D array, shape (N_lat, N_lon, 3)
        Local Up unit vectors in PA coordinates.
    X, Y, Z : 2D arrays
        Surface coordinates for the 3D plot.
    """
    lat_vals = np.arange(-90.0, 90.0 + 0.5*dlat, dlat)
    lon_vals = np.arange(-180.0, 180.0 + 0.5*dlon, dlon)

    Lon_grid, Lat_grid = np.meshgrid(lon_vals, lat_vals)

    lat_rad = np.deg2rad(Lat_grid)
    lon_rad = np.deg2rad(Lon_grid)

    # For a spherical Moon, local Up is the radial unit vector.
    up_hat_grid = np.stack(
        [
            np.cos(lat_rad) * np.cos(lon_rad),
            np.cos(lat_rad) * np.sin(lon_rad),
            np.sin(lat_rad)
        ],
        axis=-1
    )

    r_grid = R_body * up_hat_grid

    X = r_grid[..., 0]
    Y = r_grid[..., 1]
    Z = r_grid[..., 2]

    return Lat_grid, Lon_grid, r_grid, up_hat_grid, X, Y, Z


def get_satellite_positions_at_elapsed_time(sat_states_pa, target_elapsed_time):
    """
    Get each satellite's PA position at the sample nearest to target_elapsed_time.
    This is robust even if different satellites have slightly different time arrays.
    """
    sat_positions_pa = {}
    sat_sample_indices = {}
    sat_sample_times = {}

    for sat_id, data in sat_states_pa.items():
        t_elapsed_sat = data["t_elapsed"]
        k_nearest = int(np.argmin(np.abs(t_elapsed_sat - target_elapsed_time)))

        sat_positions_pa[sat_id] = data["r_pa"][k_nearest]
        sat_sample_indices[sat_id] = k_nearest
        sat_sample_times[sat_id] = t_elapsed_sat[k_nearest]

    return sat_positions_pa, sat_sample_indices, sat_sample_times


def compute_global_coverage_count_grid(
    sat_positions_pa,
    r_grid,
    up_hat_grid,
    elev_mask_deg=5.0
):
    """
    Compute number of visible satellites at every latitude/longitude grid point.

    A satellite is visible if its elevation angle is greater than elev_mask_deg.
    Since local Up is known at each surface point,

        sin(elevation) = rho_hat_PA dot up_hat_PA

    so we do not need to build an ENU frame at every grid point.
    """
    coverage_count = np.zeros(r_grid.shape[:2], dtype=int)

    for sat_id, r_sat_pa in sat_positions_pa.items():
        rho_pa = r_sat_pa[None, None, :] - r_grid
        rho_norm = np.linalg.norm(rho_pa, axis=-1)
        rho_hat_pa = rho_pa / rho_norm[..., None]

        up_component = np.sum(rho_hat_pa * up_hat_grid, axis=-1)
        up_component = np.clip(up_component, -1.0, 1.0)

        elevation_deg = np.rad2deg(np.arcsin(up_component))
        is_visible = elevation_deg > elev_mask_deg

        coverage_count += is_visible.astype(int)

    return coverage_count


def add_site_markers_3d(
    ax,
    site_locations,
    R_body=1737.4,
    marker_offset_km=25.0,
    text_offset_km=90.0
):
    """
    Add surface sites to the 3D lunar sphere.

    The south-pole sites are shown as small colored markers only.
    Apollo 11 is labeled directly with a white background so it does not blend in.
    """
    for site in site_locations:
        x, y, z = latlon_to_xyz(
            site["lat_deg"],
            site["lon_deg"],
            radius=R_body + marker_offset_km
        )

        ax.scatter(
            [x], [y], [z],
            color=site["color"],
            marker=site["marker"],
            s=site["size"],
            edgecolor="k",
            linewidth=0.8,
            depthshade=False,
            zorder=10
        )

        # Only label Apollo 11 on the 3D Moon.
        if site["name"] == "Apollo 11":
            xt, yt, zt = latlon_to_xyz(
                site["lat_deg"],
                site["lon_deg"],
                radius=R_body + text_offset_km
            )

            ax.text(
                xt,
                yt,
                zt,
                "Apollo 11",
                fontsize=8,
                color="black",
                ha="center",
                va="center",
                zorder=11,
                bbox=dict(
                    facecolor="white",
                    edgecolor="black",
                    boxstyle="round,pad=0.25",
                    alpha=0.9
                )
            )


def add_site_markers_south_pole(
    ax,
    site_locations,
    R_body=1737.4,
    lat_max=-45.0
):
    """
    Add south-pole-region sites to the 2D south-pole cap plot.

    This version uses only numbered circles.
    No separate scatter dots are plotted, which reduces overlap and clutter.
    """
    for site in site_locations:
        if not site["show_on_south_pole"]:
            continue

        if site["lat_deg"] > lat_max:
            continue

        x, y, _ = latlon_to_xyz(
            site["lat_deg"],
            site["lon_deg"],
            radius=R_body
        )

        # Draw only a numbered circle at the actual site location.
        ax.text(
            x,
            y,
            site["short_label"],
            fontsize=9,
            color="black",
            ha="center",
            va="center",
            fontweight="bold",
            zorder=9,
            bbox=dict(
                facecolor="white",
                edgecolor=site["color"],
                linewidth=1.6,
                boxstyle="circle,pad=0.25",
                alpha=0.95
            )
        )


def plot_coverage_sphere(
    ax,
    X,
    Y,
    Z,
    coverage_count,
    title,
    cmap,
    norm,
    elev_view,
    azim_view,
    R_body=1737.4
):
    """
    Plot one 3D coverage sphere with a specified camera view.
    """
    facecolors = cmap(norm(coverage_count))

    ax.plot_surface(
        X,
        Y,
        Z,
        facecolors=facecolors,
        rstride=1,
        cstride=1,
        linewidth=0,
        antialiased=False,
        shade=False
    )

    # Add NASA candidate sites + Apollo 11.
    add_site_markers_3d(ax, site_locations, R_body=R_body)

    ax.set_box_aspect((1, 1, 1))
    ax.set_xlim([-R_body, R_body])
    ax.set_ylim([-R_body, R_body])
    ax.set_zlim([-R_body, R_body])

    ax.set_xlabel("PA x [km]", labelpad=4)
    ax.set_ylabel("PA y [km]", labelpad=4)
    ax.set_zlabel("PA z [km]", labelpad=4)
    ax.tick_params(axis="both", which="major", labelsize=8, pad=1)
    ax.set_title(title, fontsize=15, pad=12)

    ax.view_init(elev=elev_view, azim=azim_view)


def plot_south_pole_map(
    ax,
    X,
    Y,
    Lat_grid,
    coverage_count,
    title,
    cmap,
    norm,
    R_body=1737.4,
    lat_min=-90.0,
    lat_max=-45.0
):
    """
    Plot a clean 2D south-pole projection.

    This uses the PA x-y plane viewed from below the Moon.
    The south pole is the center of the plot.
    Only latitudes from lat_min to lat_max are shown.
    """
    # Keep only the south-pole cap so the bottom row is not cluttered.
    mask = Lat_grid <= lat_max
    coverage_cap = np.ma.masked_where(~mask, coverage_count)

    pm = ax.pcolormesh(
        X,
        Y,
        coverage_cap,
        cmap=cmap,
        norm=norm,
        shading="auto"
    )

    # Draw latitude rings to make the south-pole geometry obvious.
    for lat_ring in [-80, -70, -60, -50]:
        ring_radius = R_body * np.cos(np.deg2rad(lat_ring))

        ring = plt.Circle(
            (0, 0),
            ring_radius,
            fill=False,
            color="k",
            linewidth=0.6,
            alpha=0.35
        )

        ax.add_patch(ring)

        ax.text(
            ring_radius / np.sqrt(2),
            ring_radius / np.sqrt(2),
            f"{lat_ring}°",
            fontsize=8,
            ha="left",
            va="bottom",
            color="k",
            alpha=0.65
        )

    # Add numbered south-pole candidate locations.
    add_site_markers_south_pole(
        ax,
        site_locations,
        R_body=R_body,
        lat_max=lat_max
    )

    # Limit the view to the south polar cap.
    r_max = R_body * np.cos(np.deg2rad(lat_max))
    ax.set_xlim([-r_max, r_max])
    ax.set_ylim([-r_max, r_max])

    # Reverse both axes so the sign convention is more consistent
    # with the orientation seen in the 3D views:
    #   x goes from positive to negative left-to-right
    #   y goes from positive to negative bottom-to-top
    ax.invert_xaxis()
    ax.invert_yaxis()

    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("PA x [km]  (positive → negative)", fontsize=10)
    ax.set_ylabel("PA y [km]  (positive → negative)", fontsize=10)
    ax.tick_params(labelsize=8)
    ax.set_title(title, fontsize=14, pad=10)
    ax.grid(True, alpha=0.25)

    return pm


# -----------------------------
# Build the whole-body grid
# -----------------------------
Lat_global, Lon_global, r_global_grid, up_hat_global_grid, X_global, Y_global, Z_global = build_global_body_grid(
    dlat_global,
    dlon_global,
    R_body=R_body
)

# Use the common time span covered by all satellites.
t_common_start = max(data["t_elapsed"][0] for data in sat_states_pa.values())
t_common_end = min(data["t_elapsed"][-1] for data in sat_states_pa.values())
t_common_mid = 0.5 * (t_common_start + t_common_end)

target_elapsed_times = [t_common_start, t_common_mid, t_common_end]
coverage_maps_global = []

for snapshot_name, target_elapsed_time in zip(time_snapshot_names, target_elapsed_times):
    sat_positions_pa, sat_sample_indices, sat_sample_times = get_satellite_positions_at_elapsed_time(
        sat_states_pa,
        target_elapsed_time
    )

    coverage_count = compute_global_coverage_count_grid(
        sat_positions_pa,
        r_global_grid,
        up_hat_global_grid,
        elev_mask_deg=elev_mask_global_deg
    )

    coverage_maps_global.append(coverage_count)

    print(f"{snapshot_name} time: target = {(target_elapsed_time - t_common_start)/3600.0:.2f} hr")
    print(f"  min visible satellites over whole body: {np.nanmin(coverage_count)}")
    print(f"  max visible satellites over whole body: {np.nanmax(coverage_count)}")
    print(f"  mean visible satellites over whole body: {np.nanmean(coverage_count):.2f}\n")


# -----------------------------
# Plot: top row 3D, bottom row south-pole 2D
# -----------------------------
vmin_coverage = 4
vmax_coverage = max(int(np.max(cov_map)) for cov_map in coverage_maps_global)

# Avoid problems if the max coverage is below the chosen vmin.
vmax_coverage = max(vmin_coverage, vmax_coverage)

cmap = cm.get_cmap("viridis").copy()
cmap.set_under("black")   # anything below vmin will be black

norm = Normalize(vmin=vmin_coverage, vmax=vmax_coverage, clip=False)

fig = plt.figure(figsize=(19, 11.5))
axes_for_colorbar = []

# Top row: isometric 3D views
for i, (coverage_count, snapshot_name, target_elapsed_time) in enumerate(
    zip(coverage_maps_global, time_snapshot_names, target_elapsed_times),
    start=1
):
    ax = fig.add_subplot(2, 3, i, projection="3d")
    axes_for_colorbar.append(ax)

    elapsed_hr = (target_elapsed_time - t_common_start) / 3600.0
    title = f"{snapshot_name}\nElapsed time = {elapsed_hr:.2f} hr"

    plot_coverage_sphere(
        ax,
        X_global,
        Y_global,
        Z_global,
        coverage_count,
        title,
        cmap,
        norm,
        elev_view=25,
        azim_view=135,
        R_body=R_body
    )

# Bottom row: clean 2D south-pole cap views
for i, (coverage_count, snapshot_name) in enumerate(
    zip(coverage_maps_global, time_snapshot_names),
    start=4
):
    ax = fig.add_subplot(2, 3, i)
    axes_for_colorbar.append(ax)

    plot_south_pole_map(
        ax,
        X_global,
        Y_global,
        Lat_global,
        coverage_count,
        f"{snapshot_name} South Pole Cap",
        cmap,
        norm,
        R_body=R_body,
        lat_min=-90.0,
        lat_max=-45.0
    )

fig.subplots_adjust(
    left=0.05,
    right=0.90,
    bottom=0.07,
    top=0.86,
    wspace=0.20,
    hspace=0.32
)

# Shared colorbar across all six subplots.
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cax = fig.add_axes([0.92, 0.22, 0.013, 0.56])

cbar = fig.colorbar(
    sm,
    cax=cax,
    ticks=np.arange(vmin_coverage, vmax_coverage + 1, 1),
    extend="min"
)

cbar.set_label(
    f"Number of visible satellites\nabove {elev_mask_global_deg:.1f} deg",
    fontsize=11
)

# Figure-level legend for surface sites.
legend_handles = [
    Line2D(
        [0],
        [0],
        marker=site["marker"],
        color="w",
        label=f'{site["short_label"]}: {site["name"]}',
        markerfacecolor=site["color"],
        markeredgecolor="k",
        markersize=9
    )
    for site in site_locations
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.46, 0.985),
    ncol=5,
    frameon=True,
    fontsize=9
)

fig.suptitle(
    "Whole-Body Coverage and South Pole Coverage at Start, Middle, and End",
    fontsize=18,
    y=0.935
)

plt.show()